# score_runs_omnicontext_sp_iii_step2000
Scores `sp_iii_p139__step2000` — Soft prompting variant (iii), prefix K/V at p=139, checkpoint 2000 — the parameter-matched point against LoRA rank 4. Scored with GPT-4.1 through OmniContext's own `OmniContextScore`, the same path every domain arm used. One difference from the domain scorers: `with_scene` follows `test_omnicontext_score.py`, which sets it from `task_type.find('scene') != -1`. The flag switches both the PF and the SC rubric, so the 150 scene rows are judged under a different rule than the other 250. Domain rows carry no scene image and run `with_scene=False` throughout.

Reads `omnicontext/manifest.jsonl` and `omnicontext/images/`, and the generated images from `gen/omnicontext/{RUN}`. Rows are indexed `{i:05d}.png` against the frozen manifest order, and alignment is asserted on `key` rather than trusted.

Reports pooled means over the 400 rows, which is the basis the domain headline uses, and the macro-average over the 8 subtasks, which is the basis the published leaderboard uses. CPU runtime; no GPU, no compute units.

Paired against the 512 baseline on the same 400 rows. GEN_NAME carries the double underscore the soft-prompt generation notebook writes.

## 1. Clone, patch the scorer

In [1]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","regex"], check=True)   # json_util imports `regex`
sys.path.insert(0, REPO)
os.chdir(REPO)

# without the break, evaluate() re-runs a successful parse to max_tries: 6 calls per row instead of 2
p = "/content/OmniGen2/omnicontext/omnicontext_score.py"
lines = open(p).read().split("\n")
if not any("added: stop after successful parse" in l for l in lines):
    out = []
    for ln in lines:
        out.append(ln)
        if 'raise Exception("rate_limit_exceeded")' in ln:
            out.append("                break   # added: stop after successful parse")
    open(p, "w").write("\n".join(out))
    print("patched")
else:
    print("already patched")

patched


## 2. Drive, run, scores

In [2]:
from google.colab import drive, userdata
import os, json
drive.mount("/content/drive")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

DRIVE    = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
OC       = f"{DRIVE}/omnicontext"
DATASET  = "omnicontext"       # the 400-row public benchmark, 8 subtasks x 50
RUN      = "sp_iii_p139_step2000"
GEN_NAME = "sp_iii_p139__step2000"
BASE_RUN = "baseline"

GEN    = f"{DRIVE}/gen/{DATASET}/{GEN_NAME}"
SCORES = f"{DRIVE}/scores/{DATASET}__{RUN}.jsonl"
os.makedirs(os.path.dirname(SCORES), exist_ok=True)

have = sorted(int(f[:-4]) for f in os.listdir(GEN) if f.endswith(".png") and f[:-4].isdigit())
assert have, f"no generated images in {GEN}"
print(f"images : {GEN} -> {len(have)} (rows {have[0]}..{have[-1]})")
print(f"scores : {SCORES}")
def geom(man):
    # the baseline notebooks record width/height/align_res under "geometry"; the arm notebooks,
    # which descend from the domain generators, record size/align_res under "sampler"
    g, s = man.get("geometry") or {}, man.get("sampler") or {}
    return (g.get("width", s.get("size")), g.get("height", s.get("size")),
            g.get("align_res", s.get("align_res")))

m = json.load(open(f"{GEN}/manifest.json")) if os.path.exists(f"{GEN}/manifest.json") else {}
if m:
    w, h, ar = geom(m)
    print(f"manifest: {m.get('gpu')} | torch {m.get('torch')} | dataset_revision {str(m.get('dataset_revision'))[:12]}")
    print(f"geometry: {w}x{h} align_res={ar}")
    print(f"negative: {'shipped default' if str(m.get('negative_prompt','')).startswith('(((deformed)))') else 'CUSTOM'}")
    bman = f"{DRIVE}/gen/{DATASET}/{BASE_RUN}/manifest.json"
    if RUN != BASE_RUN and os.path.exists(bman):
        b = json.load(open(bman))
        bw, bh, bar = geom(b)
        same = (w, h, ar, m.get("gpu"), m.get("torch")) == (bw, bh, bar, b.get("gpu"), b.get("torch"))
        print(f"vs {BASE_RUN}: {bw}x{bh} align_res={bar} | {b.get('gpu')} | torch {b.get('torch')}")
        print("  " + ("engine matches the baseline; the delta is the adapter" if same else
                      "*** ENGINE DIFFERS FROM THE BASELINE — the delta is not the adapter alone ***"))
else:
    print("manifest: absent")

Mounted at /content/drive
images : /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/omnicontext/sp_iii_p139__step2000 -> 400 (rows 0..399)
scores : /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/omnicontext__sp_iii_p139_step2000.jsonl
manifest: NVIDIA A100-SXM4-40GB | torch 2.6.0+cu124 | dataset_revision 0f6aabbe8d34
geometry: 512x512 align_res=False
negative: shipped default
vs baseline: 512x512 align_res=False | NVIDIA A100-SXM4-40GB | torch 2.6.0+cu124
  engine matches the baseline; the delta is the adapter


## 3. References and manifest

In [3]:
TAR      = f"{OC}/images.tar"
LOCALTAR = "/content/oc_images.tar"
LOCAL    = "/content/oc"
IMGS     = f"{LOCAL}/images"

rows     = [json.loads(l) for l in open(f"{OC}/manifest.jsonl", encoding="utf-8")]
REVISION = open(f"{OC}/revision.txt").read().strip()
EXPECTED = sum(len(r["input_images"]) for r in rows)

def n_files(d): return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if os.path.exists(TAR):
    subprocess.run(["cp", TAR, LOCALTAR], check=True)
else:
    # tar exits non-zero writing into the Drive FUSE mount; build locally, copy the one file up
    r = subprocess.run(["tar", "-cf", LOCALTAR, "-C", OC, "images"], capture_output=True, text=True)
    assert os.path.exists(LOCALTAR), r.stderr[-800:]
    subprocess.run(["cp", LOCALTAR, TAR], check=True)
if n_files(IMGS) < EXPECTED:
    os.makedirs(LOCAL, exist_ok=True)
    subprocess.run(["tar", "-xf", LOCALTAR, "-C", LOCAL], check=True)

assert n_files(IMGS) == EXPECTED, (n_files(IMGS), EXPECTED)
assert max(have) < len(rows), f"image index {max(have)} outside {DATASET} ({len(rows)} rows)"
missing = [p for i in have for p in rows[i]["input_images"] if not os.path.exists(os.path.join(IMGS, p))]
assert not missing, f"{len(missing)} reference images missing, first: {missing[:3]}"
if m.get("dataset_revision"):
    assert m["dataset_revision"] == REVISION, (m["dataset_revision"], REVISION)
print(f"{DATASET}: {len(rows)} rows | scoring {len(have)} | revision {REVISION[:12]} matches the run")

omnicontext: 400 rows | scoring 400 | revision 0f6aabbe8d34 matches the run


## 4. Scorer, with_scene, three-row probe
`resize_ref` and the image order are `test_omnicontext_score.py` verbatim: references capped at 1024^2 then floored to /16, generated image untouched, `input_images + [output_image]`.

In [4]:
import math, collections
from PIL import Image
from omnicontext.omnicontext_score import OmniContextScore

scorer = OmniContextScore("https://api.openai.com/v1/chat/completions", userdata.get("OPENAI_API_KEY"))

def resize_ref(im):   # OmniContext default: cap at 1024^2, round to /16 — references only
    w, h = im.size
    cap = 1024*1024
    if w*h > cap:
        r = math.sqrt(cap/(w*h)); w, h = int(w*r), int(h*r)
    return im.resize((max(16, w//16*16), max(16, h//16*16)))

def with_scene(task_type):   # test_omnicontext_score.py: task_type.find('scene') != -1
    return task_type.find("scene") != -1

_split = collections.Counter(with_scene(r["task_type"]) for r in rows)
print(f"with_scene True {_split[True]} rows | False {_split[False]} rows")
print("subtasks:", dict(sorted(collections.Counter(r["task_type"] for r in rows).items())))

# one row from each branch, so both rubrics are exercised before 400 rows are spent
_probe = [next(i for i in have if not with_scene(rows[i]["task_type"])),
          next(i for i in have if with_scene(rows[i]["task_type"]))]
for i in _probe:
    r  = rows[i]
    refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
    gen  = Image.open(f"{GEN}/{i:05d}.png").convert("RGB")
    res  = scorer.evaluate(refs + [gen], r["instruction"], with_scene=with_scene(r["task_type"]))
    print(f"\nrow {i} [{r['task_type']}] with_scene={with_scene(r['task_type'])} refs={len(refs)}")
    print(f"   PF={res['PF_scores']['score']}  SC={res['SC_scores']['score']}")
    print(f"   PF: {str(res['PF_scores']['reasoning'])[:130]}")
    print(f"   SC: {str(res['SC_scores']['reasoning'])[:130]}")

with_scene True 150 rows | False 250 rows
subtasks: {'multi_character': 50, 'multi_character_object': 50, 'multi_object': 50, 'scene_character': 50, 'scene_character_object': 50, 'scene_object': 50, 'single_character': 50, 'single_object': 50}

row 0 [single_character] with_scene=False refs=1
   PF=6  SC=8
   PF: The image depicts a man crouching in a sunlit forest, taking notes in a journal, with a lizard nearby and some plants present. How
   SC: The man in the generated forest scene matches the original in key aspects: bald head, facial hair style, glasses, and basic facial

row 250 [scene_character] with_scene=True refs=2
   PF=0  SC=0
   PF: The image shows a modern kitchen with a minimalist design, white and red cabinetry, dark gray countertops, a black coffee machine,
   SC: The final image does not include the man from the reference portrait, who is supposed to be using a laptop in the kitchen. The kit


## 5. Score, resumable

In [5]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def num(x):           # fix_json can repair a bare score into [8]
    if isinstance(x, list) and x: x = x[0]
    return x if isinstance(x, (int, float)) else None

done = set()
if os.path.exists(SCORES):
    for l in open(SCORES, encoding="utf-8"):
        try:
            d = json.loads(l)
            if "error" not in d: done.add(d["i"])   # failures stay retryable
        except: pass

def score_row(i):
    r  = rows[i]
    gp = f"{GEN}/{i:05d}.png"
    if not os.path.exists(gp): return {"i": i, "error": "missing_generated"}
    refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
    gen  = Image.open(gp).convert("RGB")            # official: generated image not resized
    time.sleep(0.2)
    res = scorer.evaluate(refs + [gen], r["instruction"], with_scene=with_scene(r["task_type"]))
    pf, sc = num(res["PF_scores"]["score"]), num(res["SC_scores"]["score"])
    if pf is None or sc is None: return {"i": i, "error": "unparsable_score"}
    # evaluate() substitutes random.randint(0,10) after 3 failed parses; that is not a measurement
    if any(str(res[k]["reasoning"]).startswith("guess_if_cannot_parse") for k in ("PF_scores", "SC_scores")):
        return {"i": i, "error": "guessed_score"}
    return {"i": i, "key": r["key"], "task_type": r["task_type"],
            "n_refs": len(r["input_images"]), "with_scene": with_scene(r["task_type"]),
            "PF": pf, "SC": sc,
            "PF_reason": res["PF_scores"]["reasoning"], "SC_reason": res["SC_scores"]["reasoning"]}

todo = [i for i in have if i not in done]
print(f"to score: {len(todo)} (done: {len(done)})")
with open(SCORES, "a", encoding="utf-8") as f, ThreadPoolExecutor(max_workers=8) as ex:
    futs = [ex.submit(score_row, i) for i in todo]
    for n, fut in enumerate(as_completed(futs), 1):
        f.write(json.dumps(fut.result(), ensure_ascii=False) + "\n"); f.flush()
        if n % 25 == 0: print(f"  {n}/{len(todo)}")
print("scoring done ->", SCORES)

to score: 400 (done: 0)
  25/400
  50/400
  75/400
  100/400
  125/400
Error: Expecting ',' delimiter: line 1 column 390 (char 389)
Now fixing:  {  "score": 2,  "reasoning": "The instruction was to have the person run together with the man in a dense forest. The final image does show two men running in what looks like a dense forest, but neither of them resemble the specific individuals from the first two images (regardless of subject identity preservation). More importantly, the composition, poses, and interactions specified ("have the person run together with the man") are only loosely addressed—the figures do not reflect any of the original image prompts in terms of appearance, attire, or context. Only the general action and forest setting are correctly applied."}
Expecting ',' delimiter: line 1 column 34 (char 33), instruction='Have the person run together with the man in a dense forest.', Attempt 1 failed, retrying after 1 seconds...
  150/400
  175/400
  200/400
  225/400
  250/4

## 6. Summary
Pooled is the basis the 488-row domain headline uses; macro over the 8 subtasks is the basis the published leaderboard uses. The published column is OmniGen2 at the released 1024x1024 `align_res=True` geometry, so it is a reference point for the shipped calibration run and not a target for the 512 arms.

In [6]:
import statistics as st
from collections import defaultdict

SIGMA_ITEM = 0.362      # judge re-scoring SD on an identical image; RMS over sd_reps (90 rows x5, ddof=1)
PUBLISHED  = {"single_character": 8.05, "single_object": 7.58,
              "multi_character": 7.11, "multi_object": 7.13, "multi_character_object": 7.45,
              "scene_character": 6.38, "scene_object": 6.71, "scene_character_object": 7.04}

def load(run):
    d, errs = {}, defaultdict(int)
    for l in open(f"{DRIVE}/scores/{DATASET}__{run}.jsonl", encoding="utf-8"):
        r = json.loads(l)
        if "error" in r: errs[r["error"]] += 1; continue
        d[r["i"]] = r
    return d, errs

ov = lambda d: math.sqrt(max(d["PF"], 0) * max(d["SC"], 0))
cur, errs = load(RUN)
print(f"{DATASET}/{RUN}: {len(cur)} scored" + (f" | errors {dict(errs)}" if errs else ""))
print(f"  pooled  PF {st.mean([d['PF'] for d in cur.values()]):.3f}  "
      f"SC {st.mean([d['SC'] for d in cur.values()]):.3f}  "
      f"Overall {st.mean([ov(d) for d in cur.values()]):.3f}")

bysub = defaultdict(list)
for d in cur.values(): bysub[d["task_type"]].append(ov(d))
macro = st.mean([st.mean(v) for v in bysub.values()])
print(f"  macro over {len(bysub)} subtasks: Overall {macro:.3f}")
print(f"\n  {'subtask':26s} {'n':>4s} {'Overall':>8s} {'published':>10s} {'delta':>7s}")
for k in sorted(bysub):
    mu, pub = st.mean(bysub[k]), PUBLISHED.get(k)
    print(f"  {k:26s} {len(bysub[k]):4d} {mu:8.3f} {pub:10.2f} {mu-pub:+7.3f}")
print(f"  {'AVERAGE (macro)':26s} {len(cur):4d} {macro:8.3f} {st.mean(list(PUBLISHED.values())):10.2f} "
      f"{macro-st.mean(list(PUBLISHED.values())):+7.3f}")

for key in ["n_refs", "with_scene"]:
    g = defaultdict(list)
    for d in cur.values(): g[d[key]].append(ov(d))
    print("  by " + key + ": " + " | ".join(f"{k}: {st.mean(v):.3f} (n={len(v)})" for k, v in sorted(g.items(), key=str)))

if RUN != BASE_RUN:
    from scipy.stats import ttest_rel
    base, _ = load(BASE_RUN)
    I = sorted(set(cur) & set(base))
    # a row index means nothing on its own; the key is what identifies the benchmark case
    bad = [i for i in I if cur[i]["key"] != base[i]["key"]]
    assert not bad, f"key mismatch on rows {bad[:5]} — runs are not aligned"
    print(f"\npaired delta vs {BASE_RUN} (n={len(I)})")
    for lab, fn in [("PF", lambda d: d["PF"]), ("SC", lambda d: d["SC"]), ("Overall", ov)]:
        d = [fn(cur[i]) - fn(base[i]) for i in I]
        t, p = ttest_rel([fn(cur[i]) for i in I], [fn(base[i]) for i in I])
        print(f"  {lab:8s} mean {st.mean(d):+.3f}  SD {st.stdev(d):.3f}  "
              f"SE {st.stdev(d)/math.sqrt(len(d)):.3f}  t {t:+.2f}  p {p:.3f}")
    print("\n  Overall delta by reference count — the domain leg is 50% single-reference against")
    print("  25% here, so pooled numbers are not comparable across legs without this cut:")
    g = defaultdict(list)
    for i in I: g[cur[i]["n_refs"]].append(ov(cur[i]) - ov(base[i]))
    for k, v in sorted(g.items()):
        t, p = ttest_rel([0.0]*len(v), [-x for x in v])
        print(f"    {k}-ref  n={len(v):3d}  {st.mean(v):+.3f}  p {p:.3f}")
    g = defaultdict(list)
    for i in I: g[cur[i]["task_type"]].append(ov(cur[i]) - ov(base[i]))
    print("  Overall delta by subtask:")
    for k, v in sorted(g.items()):
        print(f"    {k:26s} n={len(v):3d}  {st.mean(v):+.3f}")
    sd    = st.stdev([ov(cur[i]) - ov(base[i]) for i in I])
    floor = math.sqrt(2) * SIGMA_ITEM
    print(f"\n  sigma_d (Overall) = {sd:.3f} | judge-only floor = {floor:.3f} | "
          f"MDE at 80% power = {2.8016*sd/math.sqrt(len(I)):.3f}")

omnicontext/sp_iii_p139_step2000: 400 scored
  pooled  PF 7.190  SC 6.295  Overall 6.528
  macro over 8 subtasks: Overall 6.528

  subtask                       n  Overall  published   delta
  multi_character              50    5.791       7.11  -1.319
  multi_character_object       50    6.584       7.45  -0.866
  multi_object                 50    7.339       7.13  +0.209
  scene_character              50    5.613       6.38  -0.767
  scene_character_object       50    6.748       7.04  -0.292
  scene_object                 50    6.253       6.71  -0.457
  single_character             50    6.903       8.05  -1.147
  single_object                50    6.993       7.58  -0.587
  AVERAGE (macro)             400    6.528       7.18  -0.653
  by n_refs: 1: 6.948 (n=100) | 2: 6.538 (n=210) | 3: 6.038 (n=90)
  by with_scene: False: 6.722 (n=250) | True: 6.205 (n=150)

paired delta vs baseline (n=400)
  PF       mean -0.385  SD 2.334  SE 0.117  t -3.30  p 0.001
  SC       mean -1.137  SD 2.